# **Nhập dữ liệu**



In [5]:
import sqlite3 as sql
import pandas as pd

# ==============================
# Khởi tạo database tạm
# ==============================
db = sql.connect(":memory:")
cur = db.cursor()

# ==============================
# Tạo bảng bằng dictionary schema
# ==============================
tables = {
    "course": """
        CREATE TABLE course(
            id INTEGER PRIMARY KEY,
            course_name TEXT NOT NULL
        )
    """,
    "student": """
        CREATE TABLE student(
            student_id INTEGER PRIMARY KEY,
            name TEXT,
            class TEXT,
            course_id INTEGER,
            score REAL
        )
    """
}

for query in tables.values():
    cur.execute(query)

# Bảng course
course_data = [
    {"id": 12, "name": "Giai tich"},
    {"id": 34, "name": "Thong ke"},
    {"id": 26, "name": "Tin hoc"}
]

cur.executemany(
    "INSERT INTO course(id, course_name) VALUES (:id, :name)",
    course_data
)


# Bảng student
student_data = [
    (1,"Nguyen Minh Hoang","May Tinh",12,6.7),
    (2,"Tran Thi Lan","Kinh Te",34,9.2),
    (3,"Pham Van Nam","Toan Tin",None,7.9),
    (4,"Le Thanh Huyen","Toan Tin",20,7.2),
    (5,"Vu Quoc Anh","May Tinh",24,8.0),
    (6,"Dang Thuy Linh","May Tinh",24,5.5),
    (7,"Bui Tien Dung","Kinh Te",34,9.2),
    (8,"Ho Ngoc Mai","Toan Tin",20,8.8),
    (9,"Duong Huu Phuc","Kinh Te",None,7.2),
    (10,"Cao Thi Hanh","May Tinh",None,7.0)
]

insert_sql = """
INSERT INTO student
(student_id, name, class, course_id, score)
VALUES (?, ?, ?, ?, ?)
"""

for row in student_data:
    cur.execute(insert_sql, row)

db.commit()

# Kiểm tra dữ liệu
print("Bang COURSE")
print(pd.read_sql("SELECT * FROM course", db))

print("\nBang STUDENT")
print(pd.read_sql("SELECT * FROM student", db))

Bang COURSE
   id course_name
0  12   Giai tich
1  26     Tin hoc
2  34    Thong ke

Bang STUDENT
   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7
1           2       Tran Thi Lan   Kinh Te       34.0    9.2
2           3       Pham Van Nam  Toan Tin        NaN    7.9
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2
4           5        Vu Quoc Anh  May Tinh       24.0    8.0
5           6     Dang Thuy Linh  May Tinh       24.0    5.5
6           7      Bui Tien Dung   Kinh Te       34.0    9.2
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2
9          10       Cao Thi Hanh  May Tinh        NaN    7.0


# **BÀI 1**: kết nối 2 bảng

## 1. Tích Descartes

In [7]:
print(" Tích Descartes")

sql_cartesian = """
SELECT
    st.student_id,
    st.name,
    st.class,
    cr.id AS course_id,
    cr.course_name,
    st.score
FROM student st
CROSS JOIN course cr
"""

cartesian_result = pd.read_sql(sql_cartesian, conn)
print(cartesian_result)

 Tích Descartes
    student_id               name     class  course_id course_name  score
0            1  Nguyen Minh Hoang  May Tinh         12   Giai tich    6.7
1            1  Nguyen Minh Hoang  May Tinh         26     Tin hoc    6.7
2            1  Nguyen Minh Hoang  May Tinh         34    Thong ke    6.7
3            2       Tran Thi Lan   Kinh Te         12   Giai tich    9.2
4            2       Tran Thi Lan   Kinh Te         26     Tin hoc    9.2
5            2       Tran Thi Lan   Kinh Te         34    Thong ke    9.2
6            3       Pham Van Nam  Toan Tin         12   Giai tich    7.9
7            3       Pham Van Nam  Toan Tin         26     Tin hoc    7.9
8            3       Pham Van Nam  Toan Tin         34    Thong ke    7.9
9            4     Le Thanh Huyen  Toan Tin         12   Giai tich    7.2
10           4     Le Thanh Huyen  Toan Tin         26     Tin hoc    7.2
11           4     Le Thanh Huyen  Toan Tin         34    Thong ke    7.2
12           5        

## **Nhận xét**: Kết quả của phép tích Descartes cho thấy dữ liệu bị lặp lại nhiều lần, khi mỗi sinh viên xuất hiện tương ứng với tất cả các môn học trong bảng, dẫn đến việc mỗi sinh viên lặp lại nhiều dòng. Điều này không phản ánh đúng mối quan hệ thực tế giữa sinh viên và môn học, vì trên thực tế mỗi sinh viên chỉ liên kết với một hoặc một số môn nhất định thông qua course_id. Do đó, phép này chủ yếu chỉ phù hợp trong các trường hợp cần tạo ra tất cả các tổ hợp dữ liệu hoặc dùng để kiểm tra logic của các phép nối (JOIN), thay vì sử dụng để biểu diễn dữ liệu thực tế.

## 2. INNER JOIN

In [8]:
print("INNER JOIN")

sql_inner = """
SELECT
    st.student_id,
    st.name,
    st.class,
    cr.course_name,
    st.score
FROM student AS st
JOIN course AS cr
ON st.course_id = cr.id
"""

inner_result = pd.read_sql(sql_inner, conn)
print(inner_result)

INNER JOIN
   student_id               name     class course_name  score
0           1  Nguyen Minh Hoang  May Tinh   Giai tich    6.7
1           2       Tran Thi Lan   Kinh Te    Thong ke    9.2
2           7      Bui Tien Dung   Kinh Te    Thong ke    9.2


## **Nhận xét**: Kết quả chỉ hiển thị các sinh viên có course_id khớp với bảng course, nên dữ liệu chính xác và không bị lặp. Các sinh viên không có hoặc không khớp course_id sẽ không xuất hiện.

## 3. LEFT JOIN

In [9]:
print("LEFT JOIN")

sql_left = """
SELECT
    st.student_id,
    st.name,
    st.class,
    st.course_id,
    cr.course_name,
    st.score
FROM student st
LEFT JOIN course cr
ON st.course_id = cr.id
"""

left_result = pd.read_sql(sql_left, conn)
print(left_result)

LEFT JOIN
   student_id               name     class  course_id course_name  score
0           1  Nguyen Minh Hoang  May Tinh       12.0   Giai tich    6.7
1           2       Tran Thi Lan   Kinh Te       34.0    Thong ke    9.2
2           3       Pham Van Nam  Toan Tin        NaN        None    7.9
3           4     Le Thanh Huyen  Toan Tin       20.0        None    7.2
4           5        Vu Quoc Anh  May Tinh       24.0        None    8.0
5           6     Dang Thuy Linh  May Tinh       24.0        None    5.5
6           7      Bui Tien Dung   Kinh Te       34.0    Thong ke    9.2
7           8        Ho Ngoc Mai  Toan Tin       20.0        None    8.8
8           9     Duong Huu Phuc   Kinh Te        NaN        None    7.2
9          10       Cao Thi Hanh  May Tinh        NaN        None    7.0


## **Nhận xét**: Kết quả của phép LEFT JOIN cho thấy tất cả các sinh viên trong bảng bên trái đều được giữ lại, kể cả những sinh viên không có mã môn học tương ứng trong bảng bên phải. Đối với các sinh viên này, các cột thông tin từ bảng môn học sẽ hiển thị giá trị NaN hoặc None. Như vậy, phép LEFT JOIN giúp giữ lại toàn bộ dữ liệu từ bảng student và vẫn thể hiện được mối liên hệ với bảng course nếu có, phản ánh dữ liệu một cách đầy đủ hơn so với INNER JOIN.

## 4. RIGHT JOIN

In [10]:
print("RIGHT JOIN")

sql_right = """
SELECT
    cr.id AS course_id,
    cr.course_name,
    st.student_id,
    st.name,
    st.class,
    st.score
FROM course cr
LEFT JOIN student st
ON cr.id = st.course_id
"""

right_result = pd.read_sql(sql_right, conn)
print(right_result)

RIGHT JOIN
   course_id course_name  student_id               name     class  score
0         12   Giai tich         1.0  Nguyen Minh Hoang  May Tinh    6.7
1         26     Tin hoc         NaN               None      None    NaN
2         34    Thong ke         7.0      Bui Tien Dung   Kinh Te    9.2
3         34    Thong ke         2.0       Tran Thi Lan   Kinh Te    9.2


## **Nhận xét**: Kết quả cho thấy tất cả các môn học từ bảng bên phải được giữ lại hoàn toàn, ngay cả khi không có sinh viên nào đăng ký. Những môn học chưa có sinh viên (như môn Tin học) sẽ hiển thị giá trị NaN hoặc None ở các cột thông tin sinh viên

## 5. FULL OUTER JOIN

In [11]:
print("FULL OUTER JOIN")

sql_full = """
SELECT
    st.student_id,
    st.name,
    st.class,
    cr.course_name,
    st.score
FROM student st
LEFT JOIN course cr
ON st.course_id = cr.id

UNION

SELECT
    st.student_id,
    st.name,
    st.class,
    cr.course_name,
    st.score
FROM course cr
LEFT JOIN student st
ON cr.id = st.course_id
"""

full_result = pd.read_sql(sql_full, conn)
print(full_result)

FULL OUTER JOIN
    student_id               name     class course_name  score
0          NaN               None      None     Tin hoc    NaN
1          1.0  Nguyen Minh Hoang  May Tinh   Giai tich    6.7
2          2.0       Tran Thi Lan   Kinh Te    Thong ke    9.2
3          3.0       Pham Van Nam  Toan Tin        None    7.9
4          4.0     Le Thanh Huyen  Toan Tin        None    7.2
5          5.0        Vu Quoc Anh  May Tinh        None    8.0
6          6.0     Dang Thuy Linh  May Tinh        None    5.5
7          7.0      Bui Tien Dung   Kinh Te    Thong ke    9.2
8          8.0        Ho Ngoc Mai  Toan Tin        None    8.8
9          9.0     Duong Huu Phuc   Kinh Te        None    7.2
10        10.0       Cao Thi Hanh  May Tinh        None    7.0


## Nhận xét: Kết quả cho thấy các dữ liệu đều gom chung vào nhau khi giữ lại toàn bộ dữ liệu từ cả hai bảng, bao gồm cả những bản ghi không có sự trùng khớp. Nó hiển thị đồng thời cả những sinh viên chưa có môn học và những môn học chưa có sinh viên đăng ký thông qua các giá trị NaN/None. Phép nối này cực kỳ hữu ích trong việc rà soát tổng thể

## **BÀI 2**

In [16]:
print("CAP NHAT COURSE_ID VA LAM SACH DU LIEU")

cursor.executescript("""

-- 1. Cập nhật course_id còn thiếu
UPDATE student
SET course_id = (
    SELECT id
    FROM course
    ORDER BY id
    LIMIT 1
)
WHERE course_id IS NULL;

-- 2. Loại bỏ sinh viên học môn không tồn tại
DELETE FROM student
WHERE course_id NOT IN (
    SELECT id FROM course
);

""")

conn.commit()

# Kiểm tra kết quả sau khi xử lý
df_clean = pd.read_sql_query("""
SELECT s.student_id,
       s.name,
       s.class,
       s.course_id,
       c.course_name,
       s.score
FROM student s
LEFT JOIN course c
ON s.course_id = c.id
ORDER BY s.student_id
""", conn)

print(df_clean)

CAP NHAT COURSE_ID VA LAM SACH DU LIEU
   student_id               name     class  course_id course_name  score
0           1  Nguyen Minh Hoang  May Tinh         12   Giai tich    6.7
1           2       Tran Thi Lan   Kinh Te         34    Thong ke    9.2
2           3       Pham Van Nam  Toan Tin         26     Tin hoc    7.9
3           7      Bui Tien Dung   Kinh Te         34    Thong ke    9.2
4           9     Duong Huu Phuc   Kinh Te         26     Tin hoc    7.2
5          10       Cao Thi Hanh  May Tinh         26     Tin hoc    7.0


## a. Tổng số sinh viên & điểm trung bình theo lớp

In [19]:
print("Tong SV va Diem TB theo LOP")

sql_a = """
SELECT
    class,
    COUNT(student_id) AS total_students,
    ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class
ORDER BY class
"""

df_a = pd.read_sql_query(sql_a, conn)
print(df_a)

Tong SV va Diem TB theo LOP
      class  total_students  avg_score
0   Kinh Te               3       8.53
1  May Tinh               2       6.85
2  Toan Tin               1       7.90


## **Nhận xét**: Việc sử dụng hàm COUNT() để thống kê số lượng và AVG() kết hợp với ROUND() cho kết quả cho thấy sự phân bổ sinh viên và năng lực học tập không đồng đều giữa các lớp. Lớp Kinh Te đang chiếm ưu thế cả về số lượng thành viên (3 SV) lẫn điểm trung bình cao nhất (8.53). Ngược lại, lớp Toan Tin có số lượng sinh viên ít nhất và lớp May Tinh có mức điểm trung bình thấp nhất trong bảng thống kê

## b. Tổng số sinh viên & điểm trung bình theo môn học

In [22]:
print("Tong SV va Diem TB theo MON HOC")

sql_b = """
SELECT
    c.course_name,
    COUNT(s.student_id) AS total_students,
    ROUND(AVG(s.score), 2) AS avg_score
FROM student s
JOIN course c
ON s.course_id = c.id
GROUP BY c.course_name
ORDER BY c.course_name
"""

df_b = pd.read_sql_query(sql_b, conn)
print(df_b)

Tong SV va Diem TB theo MON HOC
  course_name  total_students  avg_score
0   Giai tich               1       6.70
1    Thong ke               2       9.20
2     Tin hoc               3       7.37


## **Nhận xét**: Đoạn mã sử dụng JOIN và GROUP BY rất giúp kết nối dữ liệu giữa hai bảng để thống kê chi tiết theo tên môn học. Kết quả phản ánh rõ sự phân hóa: môn Thong ke dẫn đầu về chất lượng học tập (9.20 điểm), trong khi môn Tin hoc có quy mô đông nhất (3 SV). Ngược lại, môn Giai tich có điểm số thấp nhất (6.70)

## c. Phân loại thi đua theo điểm trung bình từng môn
- Điểm TB ≥ 9.0: Xuất sắc
- 6.0 - 8.9 → Tốt
- Điểm TB < 6.0 : Kém

In [25]:
print("Phan loai thi dua theo tung mon hoc")

sql_c = """
SELECT
    c.id AS course_id,
    c.course_name,
    COUNT(s.student_id) AS total_students,
    ROUND(AVG(s.score), 2) AS avg_score,

    CASE
        WHEN ROUND(AVG(s.score),2) >= 9.0 THEN 'Xuat sac'
        WHEN ROUND(AVG(s.score),2) >= 6.0 THEN 'Tot'
        ELSE 'Kem'
    END AS classification

FROM course c
LEFT JOIN student s
ON s.course_id = c.id

GROUP BY c.id, c.course_name
ORDER BY avg_score DESC
"""

df_c = pd.read_sql_query(sql_c, conn)
print(df_c)

Phan loai thi dua theo tung mon hoc
   course_id course_name  total_students  avg_score classification
0         34    Thong ke               2       9.20       Xuat sac
1         26     Tin hoc               3       7.37            Tot
2         12   Giai tich               1       6.70            Tot


## **Nhận xét**: Môn Thong ke đạt mức Xuat sac (9.20), trong khi các môn còn lại như Tin hoc và Giai tich chỉ dừng ở mức Tot.

# **Bài 3**:

## a.1) Hãy xếp hạng sinh viên theo điểm số.


In [26]:
print("Xep hang sinh vien theo diem so")

sql_rank_all = """
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    RANK() OVER (ORDER BY score DESC) AS ranking
FROM student
"""

df_rank_all = pd.read_sql_query(sql_rank_all, conn)
print(df_rank_all)

Xep hang sinh vien theo diem so
   student_id               name     class  course_id  score  ranking
0           2       Tran Thi Lan   Kinh Te         34    9.2        1
1           7      Bui Tien Dung   Kinh Te         34    9.2        1
2           3       Pham Van Nam  Toan Tin         26    7.9        3
3           9     Duong Huu Phuc   Kinh Te         26    7.2        4
4          10       Cao Thi Hanh  May Tinh         26    7.0        5
5           1  Nguyen Minh Hoang  May Tinh         12    6.7        6


## **Nhận xét**: Đoạn mã sử dụng hàm cửa sổ RANK() OVER (ORDER BY score DESC) giúp tự động hóa việc xếp hạng và xử lý chính xác các trường hợp trùng điểm bằng cách gán cùng một thứ hạng. Kết quả truy vấn phản ánh rõ nét sự phân hóa năng lực khi Tran Thi Lan và Bui Tien Dung cùng giữ vị trí dẫn đầu (hạng 1)

## a.2) Top 3 điểm cao nhất theo điểm số

In [27]:
print("Top 3 diem cao nhat")

top3_high = pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS ranking
    FROM student
)
WHERE ranking <= 3
""", conn)

print(top3_high)

Top 3 diem cao nhat
   student_id           name     class  course_id  score  ranking
0           2   Tran Thi Lan   Kinh Te         34    9.2        1
1           7  Bui Tien Dung   Kinh Te         34    9.2        1
2           3   Pham Van Nam  Toan Tin         26    7.9        3


## **Nhận xét**: Đoạn mã sử dụng truy vấn con kết hợp hàm RANK() rất tối ưu, giúp lọc chính xác nhóm dẫn đầu mà vẫn giữ lại đầy đủ các trường hợp bằng điểm nhau. Kết quả cho thấy sự vượt trội của lớp Kinh Te với hai vị trí cùng hạng nhất (9.2 điểm), theo sau là Pham Van Nam (7.9 điểm)

## a.3) Top 3 điểm thấp nhất theo điểm số

In [28]:
print("Top 3 diem thap nhat")

top3_low = pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS ranking
    FROM student
)
WHERE ranking <= 3
""", conn)

print(top3_low)

Top 3 diem thap nhat
   student_id               name     class  course_id  score  ranking
0           1  Nguyen Minh Hoang  May Tinh         12    6.7        1
1          10       Cao Thi Hanh  May Tinh         26    7.0        2
2           9     Duong Huu Phuc   Kinh Te         26    7.2        3


## **Nhận xét**: Đoạn mã sử dụng hàm RANK() OVER (ORDER BY score ASC) giúp đảo ngược thứ tự ưu tiên để xác định chính xác nhóm sinh viên có điểm số thấp nhất. Việc lồng hàm vào truy vấn con giúp lọc dữ liệu linh hoạt, đảm bảo tính công bằng nếu có trường hợp đồng hạng ở cuối bảng. Kết quả cho thấy nhóm cần chú ý chủ yếu nằm ở lớp May Tinh, với Nguyen Minh Hoang có mức điểm thấp nhất (6.7)

## b.1) Xếp hạng theo lớp

In [29]:
print("Xep hang theo lop")

sql_rank_class = """
SELECT
    student_id,
    name,
    class,
    score,
    RANK() OVER (
        PARTITION BY class
        ORDER BY score DESC
    ) AS ranking_in_class
FROM student
"""

df_rank_class = pd.read_sql_query(sql_rank_class, conn)
print(df_rank_class)

Xep hang theo lop
   student_id               name     class  score  ranking_in_class
0           2       Tran Thi Lan   Kinh Te    9.2                 1
1           7      Bui Tien Dung   Kinh Te    9.2                 1
2           9     Duong Huu Phuc   Kinh Te    7.2                 3
3          10       Cao Thi Hanh  May Tinh    7.0                 1
4           1  Nguyen Minh Hoang  May Tinh    6.7                 2
5           3       Pham Van Nam  Toan Tin    7.9                 1


## **Nhận xét**: Đoạn mã sử dụng hàm RANK() kết hợp với mệnh đề PARTITION BY class, giúp chia nhỏ dữ liệu để xếp hạng riêng biệt cho từng lớp. Kết quả cho thấy các vị trí dẫn đầu là: Tran Thi Lan và Bui Tien Dung (Kinh Te), Cao Thi Hanh (May Tinh) và Pham Van Nam (Toan Tin).

## b.2) Top 3 cao nhất theo lớp

In [31]:
print("TOP 3 CAO NHAT THEO LOP")

top3_class_high = pd.read_sql_query("""
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        score,
        RANK() OVER (
            PARTITION BY class
            ORDER BY score DESC
        ) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
ORDER BY class, rank_in_class
""", conn)

print(top3_class_high)

TOP 3 CAO NHAT THEO LOP
   student_id               name     class  score  rank_in_class
0           2       Tran Thi Lan   Kinh Te    9.2              1
1           7      Bui Tien Dung   Kinh Te    9.2              1
2           9     Duong Huu Phuc   Kinh Te    7.2              3
3          10       Cao Thi Hanh  May Tinh    7.0              1
4           1  Nguyen Minh Hoang  May Tinh    6.7              2
5           3       Pham Van Nam  Toan Tin    7.9              1


## **Nhận xét**: Đoạn mã sử dụng hàm RANK() kết hợp PARTITION BY class giúp phân tách dữ liệu để tìm ra những cá nhân xuất sắc nhất trong nội bộ từng lớp

*sinh viên dù đã lọc "Top 3" là do tập dữ liệu hiện tại quá nhỏ. Cụ thể, mỗi lớp trong danh sách chỉ có từ 1 đến 3 sinh viên, nên khi áp dụng điều kiện lấy 3 người cao điểm nhất, toàn bộ thành viên của từng lớp đều thỏa mãn và được giữ lại.*




## b.3) Top 3 thấp nhất theo lớp

In [32]:
print("TOP 3 THAP NHAT THEO LOP")

top3_class_low = pd.read_sql_query("""
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        score,
        RANK() OVER (
            PARTITION BY class
            ORDER BY score ASC
        ) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3
ORDER BY class, rank_in_class
""", conn)

print(top3_class_low)

TOP 3 THAP NHAT THEO LOP
   student_id               name     class  score  rank_in_class
0           9     Duong Huu Phuc   Kinh Te    7.2              1
1           2       Tran Thi Lan   Kinh Te    9.2              2
2           7      Bui Tien Dung   Kinh Te    9.2              2
3           1  Nguyen Minh Hoang  May Tinh    6.7              1
4          10       Cao Thi Hanh  May Tinh    7.0              2
5           3       Pham Van Nam  Toan Tin    7.9              1


## **Nhận xét**: Đoạn mã sử dụng hàm RANK() kết hợp PARTITION BY class và ORDER BY score ASC để tìm ra những sinh viên có điểm số thấp nhất trong nội bộ từng lớp.

*sinh viên dù đã lọc "Top 3" là do tập dữ liệu hiện tại quá nhỏ. Cụ thể, mỗi lớp trong danh sách chỉ có từ 1 đến 3 sinh viên, nên khi áp dụng điều kiện lấy 3 người cao điểm nhất, toàn bộ thành viên của từng lớp đều thỏa mãn và được giữ lại.*



## c.1) Xếp hạng theo mã môn học


In [34]:
print("Xep hang theo ma mon hoc")

sql_rank_course = """
SELECT
    student_id,
    name,
    course_id,
    score,
    RANK() OVER (
        PARTITION BY course_id
        ORDER BY score DESC
    ) AS ranking_in_course
FROM student
"""

df_rank_course = pd.read_sql_query(sql_rank_course, conn)
print(df_rank_course)

Xep hang theo ma mon hoc
   student_id               name  course_id  score  ranking_in_course
0           1  Nguyen Minh Hoang         12    6.7                  1
1           3       Pham Van Nam         26    7.9                  1
2           9     Duong Huu Phuc         26    7.2                  2
3          10       Cao Thi Hanh         26    7.0                  3
4           2       Tran Thi Lan         34    9.2                  1
5           7      Bui Tien Dung         34    9.2                  1


## **Nhận xét**: Đoạn mã sử dụng hàm RANK() kết hợp với PARTITION BY course_id, giúp chia nhỏ dữ liệu để xếp hạng sinh viên riêng biệt theo từng môn học thay vì so sánh chung chung. Kết quả cho thấy ở môn số 12 và 26, vị trí dẫn đầu thuộc về Nguyen Minh Hoang và Pham Van Nam, trong khi môn số 34 ghi nhận hai sinh viên cùng đạt hạng nhất với số điểm tuyệt vời 9.2

## c.2) Top 3 cao nhất theo mã môn học

In [47]:
print("TOP 3 CAO NHAT THEO MA MON HOC")

sql_top3_course = """
SELECT *
FROM (
    SELECT
        student_id,
        name,
        course_id,
        score,
        RANK() OVER (
            PARTITION BY course_id
            ORDER BY score DESC
        ) AS rank_in_course
    FROM student
    WHERE course_id IS NOT NULL
)
WHERE rank_in_course <= 3
ORDER BY course_id, rank_in_course;
"""

df_top3_course = pd.read_sql_query(sql_top3_course, conn)
print(df_top3_course)

TOP 3 CAO NHAT THEO MA MON HOC
   student_id               name  course_id  score  rank_in_course
0           1  Nguyen Minh Hoang         12    6.7               1
1           3       Pham Van Nam         26    7.9               1
2           9     Duong Huu Phuc         26    7.2               2
3          10       Cao Thi Hanh         26    7.0               3
4           2       Tran Thi Lan         34    9.2               1
5           7      Bui Tien Dung         34    9.2               1


## **Nhận xét**: Đoạn mã sử dụng truy vấn con kết hợp hàm RANK() và PARTITION BY course_id để sàng lọc những sinh viên có thành tích tốt nhất trong từng môn học cụ thể.

*sinh viên dù đã lọc "Top 3" là do tập dữ liệu hiện tại quá nhỏ. Cụ thể, mỗi lớp trong danh sách chỉ có từ 1 đến 3 sinh viên, nên khi áp dụng điều kiện lấy 3 người cao điểm nhất, toàn bộ thành viên của từng lớp đều thỏa mãn và được giữ lại.*

## c.3) Top 3 thấp nhất theo mã môn học

In [48]:
print("TOP 3 THAP NHAT THEO MA MON HOC")

sql_bottom3_course = """
SELECT *
FROM (
    SELECT
        student_id,
        name,
        course_id,
        score,
        RANK() OVER (
            PARTITION BY course_id
            ORDER BY score ASC
        ) AS rank_in_course
    FROM student
    WHERE course_id IS NOT NULL
)
WHERE rank_in_course <= 3
ORDER BY course_id, rank_in_course;
"""

df_bottom3_course = pd.read_sql_query(sql_bottom3_course, conn)
print(df_bottom3_course)

TOP 3 THAP NHAT THEO MA MON HOC
   student_id               name  course_id  score  rank_in_course
0           1  Nguyen Minh Hoang         12    6.7               1
1          10       Cao Thi Hanh         26    7.0               1
2           9     Duong Huu Phuc         26    7.2               2
3           3       Pham Van Nam         26    7.9               3
4           2       Tran Thi Lan         34    9.2               1
5           7      Bui Tien Dung         34    9.2               1


## Nhận xét: Đoạn mã sử dụng hàm RANK() kết hợp với ORDER BY score ASC bên trong mỗi nhóm môn học (PARTITION BY course_id) để đảo ngược thứ hạng, giúp tìm ra những sinh viên có điểm số thấp nhất. Kết quả cho thấy trong khi môn 26 có sự phân bậc rõ rệt từ 7.0 đến 7.9, thì môn 34 lại có mặt bằng chung rất cao khi những người "thấp nhất" vẫn đạt tới 9.2 điểm


*sinh viên dù đã lọc "Top 3" là do tập dữ liệu hiện tại quá nhỏ. Cụ thể, mỗi lớp trong danh sách chỉ có từ 1 đến 3 sinh viên, nên khi áp dụng điều kiện lấy 3 người cao điểm nhất, toàn bộ thành viên của từng lớp đều thỏa mãn và được giữ lại.*

# **Bài 4**: Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.


In [45]:
cursor.executescript("""

WITH ranked AS (
    SELECT
        student_id,
        RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
)

UPDATE student
SET graduation_date =
    datetime('now', '+' || (
        SELECT rnk
        FROM ranked
        WHERE ranked.student_id = student.student_id
    ) || ' days');

""")

conn.commit()

conn.commit()
print(" Đã bổ sung thêm Graduation Date")

df_grad = pd.read_sql_query("""
SELECT student_id, name, score, graduation_date
FROM student
ORDER BY score DESC
""", conn)

# IN KẾT QUẢ
print(df_grad.to_string(index=False))

 Đã bổ sung thêm Graduation Date
 student_id              name  score     graduation_date
          2      Tran Thi Lan    9.2 2026-04-04 13:58:02
          7     Bui Tien Dung    9.2 2026-04-04 13:58:02
          3      Pham Van Nam    7.9 2026-04-06 13:58:02
          9    Duong Huu Phuc    7.2 2026-04-07 13:58:02
         10      Cao Thi Hanh    7.0 2026-04-08 13:58:02
          1 Nguyen Minh Hoang    6.7 2026-04-09 13:58:02


## **Nhận xét**: RANK() bên trong lệnh UPDATE để tự động hóa việc gán ngày tốt nghiệp. Hàm RANK() xác định thứ hạng, còn hàm datetime('now', ...) thực hiện phép toán cộng ngày để tạo ra lộ trình tốt nghiệp ưu tiên. Kết quả cho thấy những sinh viên dẫn đầu như Tran Thi Lan và Bui Tien Dung (đồng hạng 1) có ngày tốt nghiệp sớm nhất, trong khi các sinh viên có thứ hạng thấp hơn sẽ nhận ngày tốt nghiệp lùi dần về sau